In [ ]:
from __future__ import annotations

import json
import math
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import Resampling, reproject

try:
    from scipy import ndimage
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False


NODATA_VALUE = -9999.0
ATTR_MIN = 1e-6
EPS = 1e-6
BETA = 2.0
LOG_EVERY = 2000

# 8-neighborhood moves, distance factor in pixel units.
NEIGHBORS = (
    (-1, 0, 1.0),
    (1, 0, 1.0),
    (0, -1, 1.0),
    (0, 1, 1.0),
    (-1, -1, math.sqrt(2.0)),
    (-1, 1, math.sqrt(2.0)),
    (1, -1, math.sqrt(2.0)),
    (1, 1, math.sqrt(2.0)),
)


@dataclass
class ResellerSeed:
    reseller_id: float
    attractiveness: float
    row: int
    col: int


@dataclass
class DijkstraResult:
    best_seed_idx: np.ndarray
    best_time_min: np.ndarray
    assigned_targets: int
    elapsed_sec: float


def _now() -> float:
    return time.perf_counter()


def _safe_ratio(n: float, d: float) -> float:
    return float(n / d) if d else 0.0


def _print_progress(prefix: str, assigned: int, total: int, elapsed: float) -> None:
    if total <= 0:
        return
    pct = 100.0 * assigned / total
    rate = _safe_ratio(assigned, elapsed)
    remaining = max(total - assigned, 0)
    eta_sec = _safe_ratio(remaining, max(rate, 1e-12))
    print(
        f"[{prefix}] {assigned}/{total} ({pct:.2f}%) | "
        f"{rate:.1f} px/s | ETA {eta_sec/60.0:.1f} min"
    )


def read_reference_population(pop_path: Path) -> Tuple[np.ndarray, Dict]:
    with rasterio.open(pop_path) as src:
        pop = src.read(1).astype(np.float32)
        profile = src.profile.copy()
    return pop, profile


def reproject_match_reference(
    src_path: Path,
    ref_profile: Dict,
    resampling: Resampling,
) -> np.ndarray:
    dst = np.full(
        (ref_profile["height"], ref_profile["width"]),
        np.nan,
        dtype=np.float32,
    )
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def normalize_car_share(car_share: np.ndarray) -> np.ndarray:
    arr = car_share.astype(np.float32, copy=True)
    finite = np.isfinite(arr)
    if finite.any():
        vmax = float(np.nanmax(arr[finite]))
        if vmax > 1.0 + 1e-4 and vmax <= 100.0 + 1e-4:
            arr[finite] = arr[finite] / 100.0
    arr[~finite] = 0.0
    np.clip(arr, 0.0, 1.0, out=arr)
    return arr


def sanitize_friction(arr: np.ndarray) -> np.ndarray:
    out = arr.astype(np.float32, copy=True)
    invalid = ~np.isfinite(out) | (out <= 0.0)
    out[invalid] = np.nan
    return out


def connected_components_8(mask: np.ndarray) -> np.ndarray:
    """Return labels (0 = background) for 8-connected True cells."""
    if HAS_SCIPY:
        structure = np.ones((3, 3), dtype=np.uint8)
        labels, _ = ndimage.label(mask, structure=structure)
        return labels.astype(np.int32)

    h, w = mask.shape
    labels = np.zeros((h, w), dtype=np.int32)
    current = 0
    stack: List[Tuple[int, int]] = []
    for r in range(h):
        for c in range(w):
            if not mask[r, c] or labels[r, c] != 0:
                continue
            current += 1
            labels[r, c] = current
            stack.append((r, c))
            while stack:
                rr, cc = stack.pop()
                for dr, dc, _ in NEIGHBORS:
                    nr, nc = rr + dr, cc + dc
                    if nr < 0 or nr >= h or nc < 0 or nc >= w:
                        continue
                    if not mask[nr, nc] or labels[nr, nc] != 0:
                        continue
                    labels[nr, nc] = current
                    stack.append((nr, nc))
    return labels


def load_reseller_seeds(
    gpkg_path: Path,
    layer: str,
    id_col: str,
    attractiveness_col: str,
    ref_profile: Dict,
) -> List[ResellerSeed]:
    gdf = gpd.read_file(gpkg_path, layer=layer)
    if gdf.empty:
        raise ValueError("No reseller features found in the input layer.")

    ref_crs = ref_profile["crs"]
    if gdf.crs is None:
        raise ValueError("Input GeoPackage has no CRS.")
    if gdf.crs != ref_crs:
        gdf = gdf.to_crs(ref_crs)

    ids = pd.to_numeric(gdf[id_col], errors="coerce")
    attrs = pd.to_numeric(gdf[attractiveness_col], errors="coerce")
    attrs = attrs.fillna(ATTR_MIN).clip(lower=ATTR_MIN)

    xs = gdf.geometry.x.to_numpy()
    ys = gdf.geometry.y.to_numpy()
    rows, cols = rowcol(ref_profile["transform"], xs, ys)

    h = int(ref_profile["height"])
    w = int(ref_profile["width"])

    # Keep only valid numeric IDs and in-grid points.
    tmp: Dict[Tuple[int, int], ResellerSeed] = {}
    for rid, att, r, c in zip(ids.to_numpy(), attrs.to_numpy(), rows, cols):
        if not np.isfinite(rid):
            continue
        rr = int(r)
        cc = int(c)
        if rr < 0 or rr >= h or cc < 0 or cc >= w:
            continue

        key = (rr, cc)
        candidate = ResellerSeed(
            reseller_id=float(rid),
            attractiveness=float(att),
            row=rr,
            col=cc,
        )

        # If multiple points map to same pixel, keep the most attractive one.
        prev = tmp.get(key)
        if prev is None or candidate.attractiveness > prev.attractiveness:
            tmp[key] = candidate

    seeds = list(tmp.values())
    if not seeds:
        raise ValueError("No valid reseller seeds mapped on reference grid.")
    return seeds


def _run_component_retry(
    best_adj: np.ndarray,
    best_time: np.ndarray,
    best_seed: np.ndarray,
    traversable: np.ndarray,
    friction: np.ndarray,
    comp_mask: np.ndarray,
    comp_targets: np.ndarray,
    comp_seed_idx: np.ndarray,
    seed_rows: np.ndarray,
    seed_cols: np.ndarray,
    sqrt_attr: np.ndarray,
    pixel_size_m: float,
) -> None:
    import heapq

    h, w = friction.shape
    done = np.zeros((h, w), dtype=bool)

    heap: List[Tuple[float, int, int, int]] = []
    for si in comp_seed_idx.tolist():
        r0 = int(seed_rows[si])
        c0 = int(seed_cols[si])
        if not comp_mask[r0, c0] or not traversable[r0, c0]:
            continue

        if best_adj[r0, c0] > 0.0 or best_seed[r0, c0] < 0:
            best_adj[r0, c0] = 0.0
            best_time[r0, c0] = 0.0
            best_seed[r0, c0] = si

        heapq.heappush(heap, (best_adj[r0, c0], r0, c0, si))

    remaining = int(np.count_nonzero(comp_targets & (best_seed < 0)))

    while heap and remaining > 0:
        adj, r, c, si = heapq.heappop(heap)
        if done[r, c]:
            continue
        if adj > best_adj[r, c]:
            continue
        if not comp_mask[r, c]:
            continue

        done[r, c] = True

        if comp_targets[r, c] and best_seed[r, c] >= 0:
            remaining -= 1

        inv_scale = 1.0 / sqrt_attr[si]
        t_curr = best_time[r, c]

        for dr, dc, dist_factor in NEIGHBORS:
            nr = r + dr
            nc = c + dc
            if nr < 0 or nr >= h or nc < 0 or nc >= w:
                continue
            if done[nr, nc] or not traversable[nr, nc] or not comp_mask[nr, nc]:
                continue

            edge_time = float(friction[nr, nc]) * (pixel_size_m * dist_factor)
            t_new = t_curr + edge_time
            adj_new = t_new * inv_scale

            if (adj_new + 1e-12) < best_adj[nr, nc] or (
                abs(adj_new - best_adj[nr, nc]) <= 1e-12 and t_new < best_time[nr, nc]
            ):
                best_adj[nr, nc] = adj_new
                best_time[nr, nc] = t_new
                best_seed[nr, nc] = si
                heapq.heappush(heap, (adj_new, nr, nc, si))


def run_huff_multisource_dijkstra(
    mode_name: str,
    friction_min_per_m: np.ndarray,
    traversable_mask: np.ndarray,
    target_mask: np.ndarray,
    labels: np.ndarray,
    seeds: Sequence[ResellerSeed],
    pixel_size_m: float,
    log_every: int = LOG_EVERY,
) -> Tuple[DijkstraResult, int]:
    """Single-pass optimized Huff assignment using source-weighted multi-source Dijkstra.

    Objective: maximize A / (t + EPS)^BETA, with BETA=2.
    Equivalent minimization for each source s: t / A^(1/BETA).
    """
    import heapq

    h, w = friction_min_per_m.shape

    best_adj = np.full((h, w), np.inf, dtype=np.float64)
    best_time = np.full((h, w), np.inf, dtype=np.float64)
    best_seed = np.full((h, w), -1, dtype=np.int32)
    finalized = np.zeros((h, w), dtype=bool)

    sqrt_attr = np.array([math.sqrt(max(s.attractiveness, ATTR_MIN)) for s in seeds], dtype=np.float64)
    seed_rows = np.array([s.row for s in seeds], dtype=np.int32)
    seed_cols = np.array([s.col for s in seeds], dtype=np.int32)

    # Identify labels with at least one seed. Fallback targets only these components.
    seed_labels = labels[seed_rows, seed_cols]
    max_label = int(labels.max())
    has_seed = np.zeros(max_label + 1, dtype=bool)
    has_seed[seed_labels[seed_labels > 0]] = True

    heap: List[Tuple[float, int, int, int]] = []
    for si, (r, c) in enumerate(zip(seed_rows, seed_cols)):
        if not traversable_mask[r, c]:
            continue
        prev_adj = best_adj[r, c]
        # Tie-break at source pixel: higher attractiveness preferred.
        if (0.0 < prev_adj) or (
            prev_adj == 0.0 and best_seed[r, c] >= 0 and sqrt_attr[si] > sqrt_attr[best_seed[r, c]]
        ):
            best_adj[r, c] = 0.0
            best_time[r, c] = 0.0
            best_seed[r, c] = si
            heapq.heappush(heap, (0.0, r, c, si))

    target_count = int(np.count_nonzero(target_mask & traversable_mask))
    assigned = 0
    fallback_hits = 0
    t0 = _now()
    last_log = 0

    while heap and assigned < target_count:
        adj, r, c, si = heapq.heappop(heap)
        if finalized[r, c]:
            continue
        if adj > best_adj[r, c]:
            continue

        finalized[r, c] = True

        if target_mask[r, c]:
            assigned += 1
            if assigned - last_log >= log_every:
                _print_progress(mode_name, assigned, target_count, _now() - t0)
                last_log = assigned

        inv_scale = 1.0 / sqrt_attr[si]
        t_curr = best_time[r, c]

        for dr, dc, dist_factor in NEIGHBORS:
            nr = r + dr
            nc = c + dc
            if nr < 0 or nr >= h or nc < 0 or nc >= w:
                continue
            if finalized[nr, nc] or not traversable_mask[nr, nc]:
                continue

            edge_time = float(friction_min_per_m[nr, nc]) * (pixel_size_m * dist_factor)
            t_new = t_curr + edge_time
            adj_new = t_new * inv_scale

            # Lexicographic tie-break: lower adjusted score, then lower raw time.
            if (adj_new + 1e-12) < best_adj[nr, nc] or (
                abs(adj_new - best_adj[nr, nc]) <= 1e-12 and t_new < best_time[nr, nc]
            ):
                best_adj[nr, nc] = adj_new
                best_time[nr, nc] = t_new
                best_seed[nr, nc] = si
                heapq.heappush(heap, (adj_new, nr, nc, si))

    # Component fallback: for target cells still unassigned but in a component that has seeds,
    # run a component-wide retry with only seeds from that component.
    unassigned = target_mask & traversable_mask & (best_seed < 0)
    eligible_fallback = unassigned & has_seed[labels]

    if np.any(eligible_fallback):
        unique_labels = np.unique(labels[eligible_fallback])
        for lbl in unique_labels:
            if lbl <= 0:
                continue

            comp_mask = labels == lbl
            comp_targets = eligible_fallback & comp_mask
            if not np.any(comp_targets):
                continue

            comp_seed_idx = np.where(seed_labels == lbl)[0]
            if comp_seed_idx.size == 0:
                continue

            fallback_hits += int(np.count_nonzero(comp_targets))
            _run_component_retry(
                best_adj,
                best_time,
                best_seed,
                traversable_mask,
                friction_min_per_m,
                comp_mask,
                comp_targets,
                comp_seed_idx,
                seed_rows,
                seed_cols,
                sqrt_attr,
                pixel_size_m,
            )

    elapsed = _now() - t0
    assigned_final = int(np.count_nonzero(target_mask & traversable_mask & (best_seed >= 0)))
    _print_progress(mode_name, assigned_final, target_count, elapsed)

    return DijkstraResult(best_seed, best_time, assigned_final, elapsed), fallback_hits


def seeds_to_lookup_df(seeds: Sequence[ResellerSeed]) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "reseller_id": [s.reseller_id for s in seeds],
            "attractiveness": [s.attractiveness for s in seeds],
            "row": [s.row for s in seeds],
            "col": [s.col for s in seeds],
        }
    )


def seed_idx_to_raster_values(
    best_seed_idx: np.ndarray,
    best_time_min: np.ndarray,
    seeds: Sequence[ResellerSeed],
    inhabited_mask: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    h, w = best_seed_idx.shape
    id_raster = np.full((h, w), NODATA_VALUE, dtype=np.float32)
    t_raster = np.full((h, w), NODATA_VALUE, dtype=np.float32)

    valid = inhabited_mask & (best_seed_idx >= 0) & np.isfinite(best_time_min)
    if np.any(valid):
        seed_ids = np.array([s.reseller_id for s in seeds], dtype=np.float64)
        idx = best_seed_idx[valid]
        id_raster[valid] = seed_ids[idx].astype(np.float32)
        t_raster[valid] = best_time_min[valid].astype(np.float32)

    return id_raster, t_raster


def write_output_raster(
    out_path: Path,
    ref_profile: Dict,
    bands: Sequence[np.ndarray],
) -> None:
    profile = ref_profile.copy()
    profile.update(
        driver="GTiff",
        dtype="float32",
        count=6,
        compress="lzw",
        nodata=NODATA_VALUE,
        tiled=True,
        bigtiff="IF_SAFER",
    )

    with rasterio.open(out_path, "w", **profile) as dst:
        for i, arr in enumerate(bands, start=1):
            dst.write(arr.astype(np.float32), i)


def run_huff_pipeline(
    dataset_dir: Path = Path("dataset_big"),
    layer: str = "resell_and_filling",
    id_col: str = "id_res&fil",
    attractiveness_col: str = "attractiveness",
    log_every: int = LOG_EVERY,
) -> Dict:
    ds = Path(dataset_dir)

    t_start = _now()

    pop_path = ds / "Population.tif"
    share_path = ds / "vehicles_allocation_share.tif"
    friction_walk_path = ds / "friction_walk.tif"
    friction_car_path = ds / "friction_moto.tif"
    gpkg_path = ds / "full_lpg_chain_nig_3857.gpkg"

    out_raster = ds / "huff_preferred_distributor_per_pixel_16.tif"
    out_lookup = ds / "huff_reseller_lookup_16.csv"
    out_profile = ds / "huff_run_profile_16.json"

    timings: Dict[str, float] = {}

    t0 = _now()
    population, ref_profile = read_reference_population(pop_path)
    inhabited_mask = np.isfinite(population) & (population > 0.0)
    timings["read_population"] = _now() - t0

    t0 = _now()
    car_share_raw = reproject_match_reference(share_path, ref_profile, Resampling.bilinear)
    walk_friction = reproject_match_reference(friction_walk_path, ref_profile, Resampling.bilinear)
    car_friction = reproject_match_reference(friction_car_path, ref_profile, Resampling.bilinear)

    car_share = normalize_car_share(car_share_raw)
    walk_share = 1.0 - car_share

    walk_friction = sanitize_friction(walk_friction)
    car_friction = sanitize_friction(car_friction)
    timings["reproject_and_sanitize"] = _now() - t0

    if ref_profile["crs"] is None:
        raise ValueError("Population raster has no CRS.")
    if str(ref_profile["crs"]).lower() not in {"epsg:3857", "3857"}:
        print(f"Warning: reference CRS is {ref_profile['crs']}, expected EPSG:3857 for metric distances.")

    t0 = _now()
    seeds_all = load_reseller_seeds(
        gpkg_path=gpkg_path,
        layer=layer,
        id_col=id_col,
        attractiveness_col=attractiveness_col,
        ref_profile=ref_profile,
    )
    timings["load_resellers"] = _now() - t0

    walk_traversable = np.isfinite(walk_friction) & (walk_friction > 0.0)
    car_traversable = np.isfinite(car_friction) & (car_friction > 0.0)

    seeds_walk = [s for s in seeds_all if walk_traversable[s.row, s.col]]
    seeds_car = [s for s in seeds_all if car_traversable[s.row, s.col]]

    if not seeds_walk:
        raise ValueError("No reseller seed falls on traversable walk cells.")
    if not seeds_car:
        raise ValueError("No reseller seed falls on traversable car cells.")

    # Save lookup over unique seeds mapped on grid and valid for at least one mode.
    lookup_seeds = {}
    for s in seeds_walk + seeds_car:
        lookup_seeds[(s.row, s.col)] = s
    seeds_lookup = list(lookup_seeds.values())
    seeds_to_lookup_df(seeds_lookup).to_csv(out_lookup, index=False)

    # Pixel size in meters (EPSG:3857, affine in meters).
    px_x = abs(float(ref_profile["transform"].a))
    px_y = abs(float(ref_profile["transform"].e))
    pixel_size_m = 0.5 * (px_x + px_y)

    t0 = _now()
    walk_labels = connected_components_8(walk_traversable)
    car_labels = connected_components_8(car_traversable)
    timings["connected_components"] = _now() - t0

    print(
        f"Targets inhabited: {int(np.count_nonzero(inhabited_mask)):,} | "
        f"Seeds walk: {len(seeds_walk):,} | Seeds car: {len(seeds_car):,}"
    )

    t0 = _now()
    walk_result, fallback_walk = run_huff_multisource_dijkstra(
        mode_name="walk",
        friction_min_per_m=walk_friction,
        traversable_mask=walk_traversable,
        target_mask=inhabited_mask,
        labels=walk_labels,
        seeds=seeds_walk,
        pixel_size_m=pixel_size_m,
        log_every=log_every,
    )
    timings["walk_assignment"] = _now() - t0

    t0 = _now()
    car_result, fallback_car = run_huff_multisource_dijkstra(
        mode_name="car",
        friction_min_per_m=car_friction,
        traversable_mask=car_traversable,
        target_mask=inhabited_mask,
        labels=car_labels,
        seeds=seeds_car,
        pixel_size_m=pixel_size_m,
        log_every=log_every,
    )
    timings["car_assignment"] = _now() - t0

    walk_id, walk_time = seed_idx_to_raster_values(
        walk_result.best_seed_idx,
        walk_result.best_time_min,
        seeds_walk,
        inhabited_mask,
    )
    car_id, car_time = seed_idx_to_raster_values(
        car_result.best_seed_idx,
        car_result.best_time_min,
        seeds_car,
        inhabited_mask,
    )

    # Copy car mode where walk is missing but car exists.
    copy_from_car = inhabited_mask & (walk_id == NODATA_VALUE) & (car_id != NODATA_VALUE)
    copied_car_mode_count = int(np.count_nonzero(copy_from_car))
    if copied_car_mode_count:
        walk_id[copy_from_car] = car_id[copy_from_car]
        walk_time[copy_from_car] = car_time[copy_from_car]

    # Apply nodata outside inhabited mask for share bands too.
    car_band = np.full_like(car_share, NODATA_VALUE, dtype=np.float32)
    walk_band = np.full_like(walk_share, NODATA_VALUE, dtype=np.float32)
    car_band[inhabited_mask] = car_share[inhabited_mask].astype(np.float32)
    walk_band[inhabited_mask] = walk_share[inhabited_mask].astype(np.float32)

    write_output_raster(
        out_raster,
        ref_profile,
        bands=(
            car_band,
            walk_band,
            walk_id,
            walk_time,
            car_id,
            car_time,
        ),
    )

    inhabited_n = int(np.count_nonzero(inhabited_mask))
    assigned_walk = int(np.count_nonzero(inhabited_mask & (walk_id != NODATA_VALUE)))
    assigned_car = int(np.count_nonzero(inhabited_mask & (car_id != NODATA_VALUE)))

    total_elapsed = _now() - t_start

    run_stats = {
        "run_date": time.strftime("%Y-%m-%d"),
        "inputs": {
            "dataset_dir": str(ds),
            "population": str(pop_path.name),
            "car_share": str(share_path.name),
            "friction_walk": str(friction_walk_path.name),
            "friction_car": str(friction_car_path.name),
            "resellers": str(gpkg_path.name),
            "layer": layer,
        },
        "parameters": {
            "beta": BETA,
            "epsilon": EPS,
            "attr_min": ATTR_MIN,
            "nodata": NODATA_VALUE,
            "pixel_size_m": pixel_size_m,
        },
        "counts": {
            "inhabited_pixels": inhabited_n,
            "seeds_walk": len(seeds_walk),
            "seeds_car": len(seeds_car),
            "assigned_walk": assigned_walk,
            "assigned_car": assigned_car,
            "fallback_walk_pixels": int(fallback_walk),
            "fallback_car_pixels": int(fallback_car),
            "copied_car_mode_pixels": copied_car_mode_count,
            "unassigned_after_fallback": int(
                np.count_nonzero(inhabited_mask & (walk_id == NODATA_VALUE) & (car_id == NODATA_VALUE))
            ),
        },
        "percentages": {
            "walk_assignment_pct": 100.0 * _safe_ratio(assigned_walk, inhabited_n),
            "car_assignment_pct": 100.0 * _safe_ratio(assigned_car, inhabited_n),
            "copied_car_mode_pct": 100.0 * _safe_ratio(copied_car_mode_count, inhabited_n),
        },
        "timings_seconds": {
            **timings,
            "walk_solver_elapsed": walk_result.elapsed_sec,
            "car_solver_elapsed": car_result.elapsed_sec,
            "total": total_elapsed,
        },
        "outputs": {
            "raster": str(out_raster.name),
            "lookup_csv": str(out_lookup.name),
            "profile_json": str(out_profile.name),
        },
    }

    with out_profile.open("w", encoding="utf-8") as f:
        json.dump(run_stats, f, indent=2)

    print("Run completed.")
    print(f"Output raster: {out_raster}")
    print(f"Lookup CSV: {out_lookup}")
    print(f"Run profile: {out_profile}")

    return run_stats


# Esecuzione
stats = run_huff_pipeline(dataset_dir=Path("dataset_big"), log_every=2000)
stats